In [ ]:
#Installs
!pip install xlrd
!pip install openpyxl

In [ ]:
#Imports
import numpy as np
import pandas as pd 
import pycountry
import csv

from datetime import datetime
from contextlib import redirect_stdout #For the log files

In [ ]:
#file_name = "../input/promed/Ventas Regulares Corregido 2.xlsx"
#df = pd.read_excel(file_name, engine='openpyxl', sheet_name='9ba16c51a1fe738624fe8dd92edc9b4')
file_name = "../input/promed/ARTICULOS-RECODING.xlsx"
df = pd.read_excel(file_name, sheet_name='ARTICULOSrecoding')

In [ ]:
##1 - Article Typology
def articleTypeEncoder(table):
    try:
        
        if 'TIPO-ART' in df.columns:
            ## Special conditions
            conditions = [ table['TIPO-ART'].eq('PRO'), table['TIPO-ART'].eq('SER'), 
                          table['TIPO-ART'].eq('LIC'), table['TIPO-ART'].eq('KIT'), 
                          table['TIPO-ART'].eq('PVA'),
                          table['TIPO-ART'].eq(float('nan')), ]

            choices = [ 'PRO', 'SER', 'LIC', 'KIT', 'PVA', 'XXX']


            table['types'] = np.select(conditions, choices, default = table['TIPO-ART'])
            table['types'] = table['types'].fillna('XXX')
            c =  table['types'].astype('string')

            types = table['types'].to_numpy()
            table.drop(['types',], axis=1, inplace=True)

            print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished article type encoding')
            return types
        
        else:
            table['types'] = 'XXX'
            types = table['types'].to_numpy()
            table.drop(['types',], axis=1, inplace=True)

            print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished article type encoding')
            return types
            
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass

In [ ]:
articleTypeEncoder(df)

In [ ]:
#2 - Especialty Articles
def num_to_letter_conv(num, cl_list):
    
    if (cl_list == True):
        if num == (-1):
            letter = 'XX'
            return letter
        else:
            num_1 = int(num / 10) #Get first part
            letter_1 = chr(ord('@') + num_1 + 1)

            num_2 = num % 10 #Get 2nd part
            letter_2 = chr(ord('@') + num_2 + 1)

            letter = str(letter_1) + str(letter_2) 
            return letter
        
    else:
        if num == (-1):
            letter = 'XXX'
            return letter
        else:
            num_1 = int(num / 10) #Get first part
            letter_1 = chr(ord('@') + num_1 + 1)

            num_2 = num % 10 #Get 2nd part
            letter_2 = chr(ord('@') + num_2 + 1)
            
            letter_3 = chr(ord('@') + 1)

            letter = str(letter_1) + str(letter_2) + str(letter_3)
            return letter
        

def articleEspReencoder(table):

    try:
        # Assigning numerical values and storing in another column
        table['Especialidad_list'] = table['CLASE'] + table['CATEGORIA']
        espStr = table['Especialidad_list'].to_numpy()
        table.drop(['Especialidad_list',], axis = 1, inplace = True)

        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished article especialty encoding')
        return espStr

    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass

#def articleEspReencoder(table):
#
#    try:
#        # Assigning numerical values and storing in another column
#        clase =  table['CLASE'].astype('category')
#        categoria = table['CATEGORIA'].astype('category')
#        
#        codes_clase = clase.cat.codes
#        cats_clase = clase.cat.categories
#        
#        codes_categoria = categoria.cat.codes
#        cats_categoria = categoria.cat.categories
#        
#        table['Clase_list']     = [num_to_letter_conv(code, cl_list = True) for code in codes_clase]
#        table['Categoria_list'] = [num_to_letter_conv(code, cl_list = False) for code in codes_categoria]
#        table['Clase_list']     = table['Clase_list'] + table['Categoria_list']
#        
#        espStr = table['Clase_list'].to_numpy()
#        table.drop(['Clase_list',], axis = 1, inplace = True)
#        table.drop(['Categoria_list',], axis = 1, inplace = True)
#                
#        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished article especialty encoding')
#        return espStr
#    
#    except:
#        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
#        pass
#        

In [ ]:
articleEspReencoder(df)

In [ ]:
##3 - Article Provider Number
def articleProviderNumber(table):
    try:
        ## Special conditions
        conditions = [ table['SecuencialPRO'].eq(float('nan')), ]
    
        choices = ['PXXXXXX']
        
        table['provider_nums'] = np.select(conditions, choices, default=table['SecuencialPRO'])
        table['provider_nums'] = table['provider_nums'].fillna('PXXXXXX')
        table['provider_nums'] =  table['provider_nums'].astype('string')
        
        number = table['provider_nums'].to_numpy()
        table.drop(['provider_nums',], axis=1, inplace=True)
        
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished article provider num encoding')
        return number
    
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass

In [ ]:
articleProviderNumber(df)

In [ ]:
#4 - Alias Marca Articles
def num_to_letter_conv(num):
    if num == (-1):
        letter = 'XXXX'
        return letter
    else:    
        num = str(num).zfill(4)

        digits = [int(digit) for digit in str(num)]
        letter = [chr(ord('@') + digit + 1) for digit in digits]
        letters = ''
        letters = letters.join(letter)
        return letters

       

def articleAliasMarcaReencoder(table):

    try:
        # Assigning numerical values and storing in another column
        clase =  table['Marca CORREGIDA'].astype('category')
        
        codes = clase.cat.codes
        cats  = clase.cat.categories
        
        table['Clase_list'] = [num_to_letter_conv(code) for code in codes]
        
        espStr = table['Clase_list'].to_numpy()
        table.drop(['Clase_list',], axis=1, inplace=True)
                
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished article alias marca encoding')
        return espStr
    
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass
        

In [ ]:
articleAliasMarcaReencoder(df)

In [ ]:
#5 - Marca Code for Articles
def marcaNumExtractor(table):

    try:
        # Extracting the numerical values and storing in a list
        codesStr_list = []
        codesStr = table['KBOXcode-MARCA']
        
        codesStr_list = [code.split('-')[0] for code in codesStr]
        
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished marca num encoding extraction')
        return codesStr_list
    
    except:    
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass


In [ ]:
#marcaNumExtractor(df)

In [115]:
#6 - Provider Article Code
def articleProviderCode(table):
    try:
        # Assigning numerical values and storing in another column

        num_article =  table['NO_ARTI'].astype('str').map(lambda num: num.replace('-', ''))
        codesStr = 'NP' + num_article.astype('string').str.zfill(18)

        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished provider article code encoding')
        return codesStr

    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass

In [116]:
articleProviderCode(df)

09-08-2021 08:48:50 Finished provider article code encoding


0        NP000000000000019207
1        NP000000000000019304
2        NP000000000000019306
3        NP000000000000019509
4        NP000000000000019514
                 ...         
90342    NP0000000000000NPA71
90343    NP0000000000000NPA75
90344    NP000000000000NPB14A
90345    NP0000000000000NPCU4
90346    NP0000000000000NPE14
Name: NO_ARTI, Length: 90347, dtype: string

In [117]:
def articleEncoder(table):
    try:
        with open('logfile.txt', 'a') as f:
            with redirect_stdout(f):                
                
                table['Codigo_KBOX_Articulo'] = articleTypeEncoder(table) + '-' + articleEspReencoder(table) + '-' + marcaNumExtractor(table) + '-' + articleProviderCode(table)

                print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished article complete sequence encoding. Undefined')
                return table['Codigo_KBOX_Articulo']
                    
    except: 
        with open('logfile.txt', 'a') as f:
            with redirect_stdout(f):
                print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass  

In [118]:
articleEncoder(df)

0        XXX-SMIV2-M0001-NP000000000000019207
1        XXX-SMCI1-M0001-NP000000000000019304
2        XXX-SMCI1-M0001-NP000000000000019306
3        XXX-SMCI1-M0001-NP000000000000019509
4        XXX-SMCI1-M0001-NP000000000000019514
                         ...                 
90342    XXX-SMEQ0-M741A-NP0000000000000NPA71
90343    XXX-SMEQ0-M741A-NP0000000000000NPA75
90344    XXX-SMEQ0-M741A-NP000000000000NPB14A
90345    XXX-SMEQ0-M741A-NP0000000000000NPCU4
90346    XXX-SMEQ0-M741A-NP0000000000000NPE14
Name: Codigo_KBOX_Articulo, Length: 90347, dtype: string

In [119]:
articleEncoder(df)
df.to_excel("recoded_articles.xlsx", sheet_name='Recoded', index = False)

In [ ]:
df